# UTS BIG DATA LANJUT — SOAL 4 (VERSI BARU)

**Spark Structured Streaming — Wikimedia / Kafka**  
**IP Master terakhir dari pengguna: `192.168.1.24`** · Topik: `wiki.recentchange.raw` · Kernel **BigDataLab**

Notebook ini dibuat **dari awal**, bukan menyambung SparkContext notebook lama yang sudah berhenti. Kerjakan **per sel**; **jangan Run All**.

**Sebelum mulai:** (1) Buka notebook baru ini di JupyterLab yang berjalan di **PC Master**. (2) Pastikan di `http://192.168.1.24:8080` Spark Master aktif dan Workers berstatus `ALIVE`. (3) Pastikan broker Kafka berjalan di `192.168.1.24:9092`, dan `advertised.listeners` mengarah ke alamat yang dapat dijangkau semua node. (4) Pilih **Kernel → Restart Kernel** untuk notebook BARU ini jika sempat membuat SparkSession menggunakan IP lama.

### Bagian UTS
- **4A:** `readStream` Kafka, `from_json`, event-time, window 1 menit, watermark 2 menit, total edit & rasio bot; output tercetak dari `foreachBatch` dan disimpan ke SQLite dashboard.
- **4B:** Perbandingan metrik `lastProgress` untuk trigger 10 detik dan 30 detik, serta ekspor CSV.
- **4C:** Stop → restart dengan **checkpoint yang sama**, audit kesinambungan offset, dan cek duplikat output SQLite.

> **PENTING — checkpoint di cluster banyak PC:** default `file:///...` hanya untuk percobaan awal. File lokal di Master **tidak otomatis** tersedia bagi seluruh Worker. Untuk bukti 4C lintas Worker yang sah, set `CHECKPOINT_ROOT` ke URI penyimpanan bersama **yang benar-benar tersedia** (mis. HDFS/S3A atau penyimpanan terdistribusi kompatibel) **sebelum mulai query pertama**. Jangan mengubah checkpoint di tengah pengujian; jangan hapus checkpoint atau database sewaktu pengujian restart.

> **PENTING — 2 menit watermark:** event yang terlambat **melebihi watermark event-time** bisa dibuang oleh Spark. Ini bukan hilang karena offset; audit 4C membedakan kesinambungan offset dengan event yang sengaja dikeluarkan oleh watermark.

## SEL 1 — Konfigurasi alamat terbaru dan folder output

In [2]:
# SEL 1 — KONFIGURASI, JALANKAN DULU
import os, json, time, sqlite3, csv, statistics, socket
from pathlib import Path
from datetime import datetime, timezone
import pyspark
from pyspark.sql import SparkSession, functions as F, types as T

MASTER_IP = "192.168.1.24"   # <- IP Master TERAKHIR dari chat
SPARK_MASTER = "local[2]"
KAFKA_BOOTSTRAP = f"{MASTER_IP}:9092"
KAFKA_TOPIC = "wiki.recentchange.raw"
APP_NAME = "UTS_S4_Wikimedia_Streaming_IP24"
SPARK_KAFKA_PACKAGE = "org.apache.spark:spark-sql-kafka-0-10_2.13:4.1.3"

# Batasi data per micro-batch agar pengujian awal tidak menelan backlog ~480k pesan sekaligus.
# Ini juga membatasi throughput benchmark; jelaskan keterbatasan ini di laporan 4B.
MAX_OFFSETS_PER_TRIGGER = 400

BASE = (Path.home() / "UTS_Soal4_IP24_LokalTes").resolve()
BASE.mkdir(parents=True, exist_ok=True)

# Isi environment variable UTS4_CHECKPOINT_URI dengan URI shared storage NYATA jika tersedia.
# Fallback di bawah HANYA percobaan, bukan bukti shared checkpoint multi-PC.
SHARED_CP_URI = os.environ.get("UTS4_CHECKPOINT_URI", "").strip()
CHECKPOINT_ROOT = SHARED_CP_URI or (BASE / "checkpoints").resolve().as_uri()

print("PYSPARK:", pyspark.__version__)
print("MASTER:", SPARK_MASTER)
print("KAFKA :", KAFKA_BOOTSTRAP, "| TOPIC:", KAFKA_TOPIC)
print("OUTPUT:", BASE)
print("CHECKPOINT:", CHECKPOINT_ROOT)
if not SHARED_CP_URI:
    print("PERINGATAN: checkpoint masih lokal; untuk bukti 4C multi-PC wajib storage bersama.")

# Tes port dasar dari PC tempat JupyterLab berjalan.
for nama, port in (("Spark Master", 7077), ("Kafka", 9092)):
    try:
        with socket.create_connection((MASTER_IP, port), timeout=4):
            print(f"[OK] {nama} {MASTER_IP}:{port} terbuka")
    except OSError as e:
        print(f"[GAGAL] {nama} {MASTER_IP}:{port}: {e}")
        raise RuntimeError("Perbaiki jaringan/layanan sesuai IP baru sebelum Sel 2.") from e

PYSPARK: 4.1.3
MASTER: local[2]
KAFKA : 192.168.1.24:9092 | TOPIC: wiki.recentchange.raw
OUTPUT: C:\Users\ASUS\UTS_Soal4_IP24_LokalTes
CHECKPOINT: file:///C:/Users/ASUS/UTS_Soal4_IP24_LokalTes/checkpoints
PERINGATAN: checkpoint masih lokal; untuk bukti 4C multi-PC wajib storage bersama.
[OK] Spark Master 192.168.1.24:7077 terbuka
[OK] Kafka 192.168.1.24:9092 terbuka


## SEL 2 — SparkSession baru + konektor Kafka

In [3]:
# SEL 2 — SPARK SESSION
if not pyspark.__version__.startswith("4.1."):
    raise RuntimeError(f"PySpark {pyspark.__version__}; cek konektor Spark-Kafka yang sesuai.")

# Kesalahan sebelumnya: SparkContext telah STOP, lalu getOrCreate() mencoba menggunakannya.
# Jika itu terjadi lagi, pakai Kernel > Restart Kernel, bukan terus menjalankan ulang sel.
old_session = SparkSession.getActiveSession()
if old_session is not None:
    if old_session.sparkContext._jsc.sc().isStopped():
        raise RuntimeError("SparkContext lama STOP. Restart Kernel notebook ini, lalu jalankan Sel 1.")
    if old_session.sparkContext.master != SPARK_MASTER:
        raise RuntimeError("SparkSession masih memakai IP lama. Restart Kernel notebook ini.")

spark = (SparkSession.builder
    .master(SPARK_MASTER)
    .appName(APP_NAME)
    .config("spark.driver.host", MASTER_IP)
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.executor.memory", "1g")
    .config("spark.cores.max", "6")
    .config("spark.jars.packages", SPARK_KAFKA_PACKAGE)
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.sql.session.timeZone", "UTC")
    .getOrCreate())

sc = spark.sparkContext
print("SPARK:", spark.version)
print("MASTER:", sc.master)
print("SPARK UI:", sc.uiWebUrl)
if sc.master != SPARK_MASTER:
    raise RuntimeError("Spark tersambung ke Master lain; restart kernel dan periksa Sel 1.")

SPARK: 4.1.3
MASTER: local[2]
SPARK UI: http://192.168.1.24:4040


## SEL 3 — Tes Spark sederhana (diagnosis sebelum streaming)

In [4]:
# SEL 3 — HARUS MENCETAK 10; jika macet, jangan lanjut ke Kafka / streaming.
print("TES SPARK dimulai ...")
n = spark.range(10, numPartitions=2).count()
print("TES SPARK SELESAI. HASIL:", n)
assert n == 10

TES SPARK dimulai ...


TES SPARK SELESAI. HASIL: 10


## SEL 4 — Baca Kafka sebagai Structured Stream (`readStream`)

In [5]:
# SEL 4 — SUMBER KAFKA
# 'earliest' digunakan agar tetap ada data saat producer sedang berhenti.
# Perhatian: jika topic berisi backlog lama, watermark 2 menit dapat membuang event-time yang terlambat.
raw_stream = (spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", KAFKA_BOOTSTRAP)
    .option("subscribe", KAFKA_TOPIC)
    .option("startingOffsets", "earliest")
    .option("failOnDataLoss", "true")
    .option("maxOffsetsPerTrigger", str(MAX_OFFSETS_PER_TRIGGER))
    .load())
print("readStream siap?", raw_stream.isStreaming)
raw_stream.printSchema()

readStream siap? True
root
 |-- key: binary (nullable = true)
 |-- value: binary (nullable = true)
 |-- topic: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- timestampType: integer (nullable = true)



## SEL 5 — `from_json` dan event time dari `timestamp` epoch detik

In [6]:
# SEL 5 — PARSING JSON
schema = T.StructType([
    T.StructField("id", T.LongType()),
    T.StructField("type", T.StringType()),
    T.StructField("wiki", T.StringType()),
    T.StructField("title", T.StringType()),
    T.StructField("user", T.StringType()),
    T.StructField("bot", T.BooleanType()),
    T.StructField("timestamp", T.LongType()),
    T.StructField("comment", T.StringType()),
])

parsed = (raw_stream
    .select(F.from_json(F.col("value").cast("string"), schema).alias("j"))
    .select("j.*"))

edits = (parsed
    .filter((F.col("type") == "edit") & F.col("wiki").isNotNull()
            & F.col("timestamp").isNotNull())
    .withColumn("event_time", F.timestamp_seconds(F.col("timestamp"))))

print("from_json + filter type=edit + event_time siap")
edits.printSchema()

from_json + filter type=edit + event_time siap
root
 |-- id: long (nullable = true)
 |-- type: string (nullable = true)
 |-- wiki: string (nullable = true)
 |-- title: string (nullable = true)
 |-- user: string (nullable = true)
 |-- bot: boolean (nullable = true)
 |-- timestamp: long (nullable = true)
 |-- comment: string (nullable = true)
 |-- event_time: timestamp (nullable = true)



## SEL 6 — Watermark 2 menit, window 1 menit, rasio bot per wiki

In [7]:
# SEL 6 — AGREGASI STREAMING
aggregated = (edits
    .withWatermark("event_time", "2 minutes")
    .groupBy(F.window(F.col("event_time"), "1 minute").alias("time_window"),
             F.col("wiki"))
    .agg(
        F.count("*").alias("total_edits"),
        F.sum(F.when(F.col("bot") == True, 1).otherwise(0)).alias("bot_edits"),
    )
    .select(
        F.col("time_window.start").alias("window_start"),
        F.col("time_window.end").alias("window_end"),
        F.col("wiki"), F.col("total_edits"), F.col("bot_edits"),
        F.round(100.0 * F.col("bot_edits") / F.col("total_edits"), 2).alias("bot_ratio_pct"),
    ))
print("WINDOW: 1 minute | WATERMARK: 2 minutes | OUTPUT: update")
aggregated.printSchema()

WINDOW: 1 minute | WATERMARK: 2 minutes | OUTPUT: update
root
 |-- window_start: timestamp (nullable = true)
 |-- window_end: timestamp (nullable = true)
 |-- wiki: string (nullable = true)
 |-- total_edits: long (nullable = false)
 |-- bot_edits: long (nullable = true)
 |-- bot_ratio_pct: double (nullable = true)



## SEL 7 — Sink `foreachBatch`: console + SQLite dashboard (idempotent)

Hasil agregasi tiap batch **dicetak ke output notebook** (sebagai console preview) dan disimpan di SQLite pada **PC Master**. Kunci unik `(wiki, window_start, window_end)` menyimpan **nilai agregasi terbaru**, bukan menambah dua kali. `batch_id` dicatat sebagai transaksi yang hanya boleh masuk sekali.

Karena Spark memiliki semantik **at-least-once** untuk `foreachBatch`, ini diperlukan agar retry batch tidak membuat hitungan dobel di dashboard.

In [8]:
# SEL 7 — FUNGSI OUTPUT, METRIK, START, STOP

def db_path(tag):
    return BASE / f"dashboard_{tag}.sqlite"

def cp_path(tag):
    return CHECKPOINT_ROOT.rstrip("/") + "/" + tag

def make_sink(tag):
    def sink(batch_df, batch_id):
        rows = batch_df.collect()  # PENTING: consume SELURUH batch untuk stateful update.
        connection = sqlite3.connect(str(db_path(tag)), timeout=60)
        try:
            connection.execute("BEGIN IMMEDIATE")
            connection.execute("""CREATE TABLE IF NOT EXISTS processed_batches (
                batch_id INTEGER PRIMARY KEY,
                number_of_rows INTEGER NOT NULL,
                saved_at_utc TEXT DEFAULT CURRENT_TIMESTAMP
            )""")
            connection.execute("""CREATE TABLE IF NOT EXISTS dashboard (
                wiki TEXT NOT NULL,
                window_start TEXT NOT NULL,
                window_end TEXT NOT NULL,
                total_edits INTEGER NOT NULL,
                bot_edits INTEGER NOT NULL,
                bot_ratio_pct REAL NOT NULL,
                last_batch_id INTEGER NOT NULL,
                PRIMARY KEY (wiki, window_start, window_end)
            )""")
            done = connection.execute(
                "SELECT 1 FROM processed_batches WHERE batch_id=?", (int(batch_id),)
            ).fetchone()
            if done:
                connection.commit()
                print(f"[{tag}] batch {batch_id}: sudah ada, SKIP (tidak dobel)")
                return
            for r in rows:
                connection.execute("""INSERT INTO dashboard
                    (wiki, window_start, window_end, total_edits,
                     bot_edits, bot_ratio_pct, last_batch_id)
                    VALUES (?, ?, ?, ?, ?, ?, ?)
                    ON CONFLICT (wiki, window_start, window_end) DO UPDATE SET
                    total_edits=excluded.total_edits,
                    bot_edits=excluded.bot_edits,
                    bot_ratio_pct=excluded.bot_ratio_pct,
                    last_batch_id=excluded.last_batch_id
                    WHERE excluded.last_batch_id > dashboard.last_batch_id""",
                    (r["wiki"], str(r["window_start"]), str(r["window_end"]),
                     int(r["total_edits"]), int(r["bot_edits"]),
                     float(r["bot_ratio_pct"] or 0), int(batch_id)))
            connection.execute(
                "INSERT INTO processed_batches (batch_id,number_of_rows) VALUES (?,?)",
                (int(batch_id), len(rows)))
            connection.commit()
            print(f"[{tag}] BATCH {batch_id} SELESAI | agregasi diperbarui: {len(rows)}")
            for r in sorted(rows, key=lambda x: x["total_edits"], reverse=True)[:5]:
                print("  ", str(r["window_start"]), r["wiki"],
                      "edit=", r["total_edits"], "bot%=", r["bot_ratio_pct"])
        except Exception:
            connection.rollback()
            raise
        finally:
            connection.close()
    return sink


def start_stream(tag, seconds):
    # Satu stream aktif dalam satu waktu agar tidak berebut resource cluster.
    running = [q.name for q in spark.streams.active]
    if running:
        raise RuntimeError(f"Masih ada streaming aktif: {running}. STOP dulu yang lama.")
    q = (aggregated.writeStream
         .queryName("UTS4_" + tag)
         .outputMode("update")
         .foreachBatch(make_sink(tag))
         .option("checkpointLocation", cp_path(tag))
         .trigger(processingTime=f"{seconds} seconds")
         .start())
    print("START:", tag, "trigger=", seconds, "detik")
    print("CHECKPOINT:", cp_path(tag), "QUERY ID:", q.id)
    return q


def save_progress(q, tag):
    fn = BASE / f"progress_{tag}.jsonl"
    seen = set()
    if fn.exists():
        for line in fn.read_text(encoding="utf-8").splitlines():
            try:
                item = json.loads(line)
                seen.add((str(item.get("id")), int(item.get("batchId", -1))))
            except (ValueError, TypeError):
                pass
    with fn.open("a", encoding="utf-8") as f:
        for p in (q.recentProgress or []):
            key = (str(p.get("id")), int(p.get("batchId", -1)))
            if key not in seen:
                f.write(json.dumps(p, default=str) + "\n")
                seen.add(key)
    return fn


def show_metrics(q, tag, watch_seconds=15):
    # Tidak otomatis stop! Kalau 0 progress, kita masih bisa melihat Job RUNNING di Spark UI.
    for t in range(0, watch_seconds, 5):
        time.sleep(5)
        print(f"detik {t+5}: aktif={q.isActive} | progress={len(q.recentProgress)}")
        if not q.isActive:
            print("ERROR query:", q.exception())
            break
    fn = save_progress(q, tag)
    p = q.lastProgress
    if p:
        data = {k:p.get(k) for k in
            ("batchId", "numInputRows", "inputRowsPerSecond",
             "processedRowsPerSecond", "durationMs", "eventTime", "sources", "stateOperators")}
        print("LAST PROGRESS:", json.dumps(data, indent=2, default=str))
        print("BUKTI LOG:", fn)
    else:
        print("BELUM ADA MICROBATCH SELESAI. Jangan buru-buru stop; cek tab Jobs/Stages/Executors saat masih aktif.")
    return p


def stop_if_progress(q, tag):
    if not q.isActive:
        print("Query sudah berhenti. Error:", q.exception())
        return None
    if not q.lastProgress:
        print("BELUM ADA progress. Tidak dihentikan agar bisa diperiksa lewat Spark UI.")
        return None
    q.stop()
    save_progress(q, tag)
    print("STOP:", tag, "| aktif?", q.isActive)
    return q.lastProgress

print("Sink SQLite, fungsi start/monitor/stop SIAP")

Sink SQLite, fungsi start/monitor/stop SIAP


## SEL 8 — 4A + 4B: START stream trigger **10 detik**

In [9]:
# SEL 8 — START 10s
q10 = start_stream("t10", 10)
print("Stream 10 detik aktif?", q10.isActive)

START: t10 trigger= 10 detik
CHECKPOINT: file:///C:/Users/ASUS/UTS_Soal4_IP24_LokalTes/checkpoints/t10 QUERY ID: 25790ec4-e16a-41e2-bfdb-bfb73c8a0edc
Stream 10 detik aktif? True


[t10] BATCH 13 SELESAI | agregasi diperbarui: 43
   2026-11-19 18:38:00 wikidatawiki edit= 209 bot%= 0.48
   2026-11-19 18:38:00 commonswiki edit= 137 bot%= 81.75
   2026-11-19 18:38:00 elwiktionary edit= 46 bot%= 100.0
   2026-11-19 18:38:00 enwiki edit= 25 bot%= 0.0
   2026-11-19 18:38:00 frwiki edit= 20 bot%= 40.0


[t10] BATCH 14 SELESAI | agregasi diperbarui: 32
   2026-11-19 18:38:00 wikidatawiki edit= 292 bot%= 0.34
   2026-11-19 18:38:00 commonswiki edit= 184 bot%= 81.52
   2026-11-19 18:38:00 elwiktionary edit= 55 bot%= 100.0
   2026-11-19 18:38:00 enwiki edit= 36 bot%= 2.78
   2026-11-19 18:38:00 frwiki edit= 28 bot%= 35.71


[t10] BATCH 15 SELESAI | agregasi diperbarui: 45
   2026-11-19 18:38:00 wikidatawiki edit= 369 bot%= 0.27
   2026-11-19 18:38:00 commonswiki edit= 208 bot%= 82.21
   2026-11-19 18:38:00 elwiktionary edit= 61 bot%= 100.0
   2026-11-19 18:38:00 enwiki edit= 42 bot%= 4.76
   2026-11-19 18:38:00 frwiki edit= 35 bot%= 28.57


[t10] BATCH 16 SELESAI | agregasi diperbarui: 46
   2026-11-19 18:38:00 wikidatawiki edit= 410 bot%= 0.24
   2026-11-19 18:38:00 commonswiki edit= 215 bot%= 81.86
   2026-11-19 18:38:00 elwiktionary edit= 68 bot%= 100.0
   2026-11-19 18:39:00 wikidatawiki edit= 51 bot%= 0.0
   2026-11-19 18:38:00 enwiki edit= 47 bot%= 4.26


[t10] BATCH 17 SELESAI | agregasi diperbarui: 42
   2026-11-19 18:38:00 wikidatawiki edit= 421 bot%= 0.24
   2026-11-19 18:38:00 commonswiki edit= 219 bot%= 81.28
   2026-11-19 18:39:00 wikidatawiki edit= 109 bot%= 0.0
   2026-11-19 18:39:00 commonswiki edit= 85 bot%= 80.0
   2026-11-19 18:38:00 elwiktionary edit= 72 bot%= 100.0


... (2 outputs left out)

[t10] BATCH 20 SELESAI | agregasi diperbarui: 50
   2026-11-19 18:39:00 wikidatawiki edit= 263 bot%= 0.0
   2026-11-19 18:39:00 commonswiki edit= 189 bot%= 79.37
   2026-11-19 18:39:00 elwiktionary edit= 68 bot%= 100.0
   2026-11-19 18:39:00 enwiki edit= 52 bot%= 7.69
   2026-11-19 18:40:00 wikidatawiki edit= 44 bot%= 0.0


[t10] BATCH 21 SELESAI | agregasi diperbarui: 47
   2026-11-19 18:39:00 wikidatawiki edit= 309 bot%= 0.0
   2026-11-19 18:39:00 commonswiki edit= 203 bot%= 79.8
   2026-11-19 18:40:00 wikidatawiki edit= 79 bot%= 0.0
   2026-11-19 18:39:00 elwiktionary edit= 72 bot%= 100.0
   2026-11-19 18:40:00 commonswiki edit= 62 bot%= 85.48


[t10] BATCH 22 SELESAI | agregasi diperbarui: 39
   2026-11-19 18:39:00 wikidatawiki edit= 330 bot%= 0.0
   2026-11-19 18:39:00 commonswiki edit= 216 bot%= 80.09
   2026-11-19 18:40:00 wikidatawiki edit= 126 bot%= 0.0
   2026-11-19 18:40:00 commonswiki edit= 99 bot%= 80.81
   2026-11-19 18:39:00 elwiktionary edit= 73 bot%= 100.0


[t10] BATCH 23 SELESAI | agregasi diperbarui: 37
   2026-11-19 18:39:00 commonswiki edit= 217 bot%= 79.72
   2026-11-19 18:40:00 wikidatawiki edit= 196 bot%= 0.0
   2026-11-19 18:40:00 commonswiki edit= 144 bot%= 78.47
   2026-11-19 18:39:00 enwiki edit= 58 bot%= 6.9
   2026-11-19 18:40:00 elwiktionary edit= 53 bot%= 100.0


## SEL 9 — Pantau `lastProgress` trigger 10 detik (boleh diulang selama query aktif)

In [10]:
# SEL 9 — MONITOR 10s
p10 = show_metrics(q10, "t10", watch_seconds=30)
# Kalau progress=0, jangan lanjut Sel 10; periksa Spark UI selagi query aktif.

[t10] BATCH 24 SELESAI | agregasi diperbarui: 44
   2026-11-19 18:40:00 wikidatawiki edit= 246 bot%= 0.0
   2026-11-19 18:40:00 commonswiki edit= 171 bot%= 80.12
   2026-11-19 18:40:00 elwiktionary edit= 63 bot%= 100.0
   2026-11-19 18:40:00 enwiki edit= 49 bot%= 6.12
   2026-11-19 18:41:00 wikidatawiki edit= 29 bot%= 0.0


detik 5: aktif=True | progress=12


detik 10: aktif=True | progress=12


[t10] BATCH 25 SELESAI | agregasi diperbarui: 46
   2026-11-19 18:40:00 wikidatawiki edit= 296 bot%= 0.0
   2026-11-19 18:40:00 commonswiki edit= 189 bot%= 80.95
   2026-11-19 18:40:00 elwiktionary edit= 68 bot%= 100.0
   2026-11-19 18:41:00 wikidatawiki edit= 62 bot%= 0.0
   2026-11-19 18:40:00 enwiki edit= 55 bot%= 5.45


detik 15: aktif=True | progress=13


detik 20: aktif=True | progress=13


[t10] BATCH 26 SELESAI | agregasi diperbarui: 48
   2026-11-19 18:40:00 wikidatawiki edit= 346 bot%= 0.0
   2026-11-19 18:40:00 commonswiki edit= 203 bot%= 81.77
   2026-11-19 18:41:00 wikidatawiki edit= 100 bot%= 0.0
   2026-11-19 18:41:00 commonswiki edit= 72 bot%= 81.94
   2026-11-19 18:40:00 elwiktionary edit= 70 bot%= 100.0


detik 25: aktif=True | progress=14


detik 30: aktif=True | progress=14
LAST PROGRESS: {
  "batchId": 26,
  "numInputRows": 398,
  "inputRowsPerSecond": 39.8,
  "processedRowsPerSecond": 164.5,
  "durationMs": {
    "addBatch": 1598,
    "commitOffsets": 515,
    "getBatch": 0,
    "latestOffset": 12,
    "queryPlanning": 25,
    "triggerExecution": 2419,
    "walCommit": 264
  },
  "eventTime": {
    "avg": "2026-11-19T11:41:10.456Z",
    "max": "2026-11-19T11:41:43.000Z",
    "min": "2026-11-19T11:40:34.000Z",
    "watermark": "2026-11-19T11:39:29.000Z"
  },
  "sources": [
    {
      "description": "KafkaV2[Subscribe[wiki.recentchange.raw]]",
      "startOffset": "{'wiki.recentchange.raw': {'0': 2756, '1': 2470, '2': 2574, '3': 2548}}",
      "endOffset": "{'wiki.recentchange.raw': {'0': 2862, '1': 2565, '2': 2673, '3': 2646}}",
      "latestOffset": "{'wiki.recentchange.raw': {'0': 127833, '1': 114825, '2': 119959, '3': 118004}}",
      "numInputRows": 398,
      "inputRowsPerSecond": 39.8,
      "processedRowsPerSeco

[t10] BATCH 27 SELESAI | agregasi diperbarui: 44
   2026-11-19 18:40:00 wikidatawiki edit= 363 bot%= 0.0
   2026-11-19 18:40:00 commonswiki edit= 209 bot%= 81.82
   2026-11-19 18:41:00 wikidatawiki edit= 174 bot%= 0.0
   2026-11-19 18:41:00 commonswiki edit= 110 bot%= 73.64
   2026-11-19 18:40:00 elwiktionary edit= 73 bot%= 100.0


[t10] BATCH 28 SELESAI | agregasi diperbarui: 40
   2026-11-19 18:41:00 wikidatawiki edit= 229 bot%= 0.0
   2026-11-19 18:41:00 commonswiki edit= 154 bot%= 68.83
   2026-11-19 18:41:00 elwiktionary edit= 42 bot%= 100.0
   2026-11-19 18:41:00 enwiki edit= 35 bot%= 8.57
   2026-11-19 18:40:00 frwiki edit= 25 bot%= 4.0


## SEL 10 — STOP trigger 10 detik dan simpan offset sebelum restart

In [11]:
# SEL 10 — STOP 10s (hanya setelah minimal 1 batch selesai)
before_restart = stop_if_progress(q10, "t10")
if before_restart:
    print("OFFSET TERAKHIR SEBELUM RESTART:")
    print(json.dumps(before_restart.get("sources", []), indent=2, default=str))

STOP: t10 | aktif? False
OFFSET TERAKHIR SEBELUM RESTART:
[
  {
    "description": "KafkaV2[Subscribe[wiki.recentchange.raw]]",
    "startOffset": "{'wiki.recentchange.raw': {'0': 2968, '1': 2660, '2': 2772, '3': 2744}}",
    "endOffset": "{'wiki.recentchange.raw': {'0': 3074, '1': 2755, '2': 2871, '3': 2842}}",
    "latestOffset": "{'wiki.recentchange.raw': {'0': 127833, '1': 114825, '2': 119959, '3': 118004}}",
    "numInputRows": 398,
    "inputRowsPerSecond": 39.8,
    "processedRowsPerSecond": 183.7,
    "metrics": {
      "avgOffsetsBehindLatest": "117269.8",
      "maxOffsetsBehindLatest": "124759",
      "minOffsetsBehindLatest": "112070"
    }
  }
]


## SEL 11 — 4B: START stream trigger **30 detik** (checkpoint terpisah)

In [12]:
# SEL 11 — START 30s
if q10.isActive:
    raise RuntimeError("q10 masih aktif. Selesaikan dulu Sel 10.")
q30 = start_stream("t30", 30)
print("Stream 30 detik aktif?", q30.isActive)

START: t30 trigger= 30 detik
CHECKPOINT: file:///C:/Users/ASUS/UTS_Soal4_IP24_LokalTes/checkpoints/t30 QUERY ID: b02f2592-518d-439e-918c-02e166959ce6
Stream 30 detik aktif? True


[t30] BATCH 0 SELESAI | agregasi diperbarui: 36
   2026-11-19 18:35:00 wikidatawiki edit= 96 bot%= 0.0
   2026-11-19 18:35:00 commonswiki edit= 44 bot%= 86.36
   2026-11-19 18:35:00 elwiktionary edit= 16 bot%= 100.0
   2026-11-19 18:35:00 enwiki edit= 14 bot%= 0.0
   2026-11-19 18:35:00 frwiki edit= 8 bot%= 12.5


[t30] BATCH 1 SELESAI | agregasi diperbarui: 30
   2026-11-19 18:35:00 wikidatawiki edit= 192 bot%= 0.0
   2026-11-19 18:35:00 commonswiki edit= 91 bot%= 85.71
   2026-11-19 18:35:00 elwiktionary edit= 31 bot%= 100.0
   2026-11-19 18:35:00 enwiki edit= 26 bot%= 0.0
   2026-11-19 18:35:00 frwiki edit= 15 bot%= 6.67


[t30] BATCH 2 SELESAI | agregasi diperbarui: 56
   2026-11-19 18:35:00 wikidatawiki edit= 219 bot%= 0.0
   2026-11-19 18:35:00 commonswiki edit= 105 bot%= 86.67
   2026-11-19 18:36:00 wikidatawiki edit= 46 bot%= 0.0
   2026-11-19 18:35:00 enwiki edit= 39 bot%= 0.0
   2026-11-19 18:35:00 elwiktionary edit= 39 bot%= 100.0


[t30] BATCH 3 SELESAI | agregasi diperbarui: 39
   2026-11-19 18:36:00 wikidatawiki edit= 95 bot%= 0.0
   2026-11-19 18:36:00 commonswiki edit= 81 bot%= 86.42
   2026-11-19 18:36:00 elwiktionary edit= 26 bot%= 100.0
   2026-11-19 18:36:00 enwiki edit= 19 bot%= 0.0
   2026-11-19 18:36:00 dewiki edit= 13 bot%= 23.08


## SEL 12 — Pantau `lastProgress` trigger 30 detik (boleh diulang)

In [13]:
# SEL 12 — MONITOR 30s
p30 = show_metrics(q30, "t30", watch_seconds=60)
# Kalau belum ada batch selesai, lihat Spark UI dan ulang SEL INI nanti; jangan langsung stop.

[t30] BATCH 4 SELESAI | agregasi diperbarui: 33
   2026-11-19 18:36:00 wikidatawiki edit= 193 bot%= 0.0
   2026-11-19 18:36:00 commonswiki edit= 128 bot%= 83.59
   2026-11-19 18:36:00 elwiktionary edit= 41 bot%= 100.0
   2026-11-19 18:36:00 enwiki edit= 32 bot%= 3.13
   2026-11-19 18:36:00 dewiki edit= 18 bot%= 22.22


detik 5: aktif=True | progress=4


detik 10: aktif=True | progress=5


detik 15: aktif=True | progress=5


detik 20: aktif=True | progress=5


detik 25: aktif=True | progress=5


... (5 outputs left out)

detik 50: aktif=True | progress=6


detik 55: aktif=True | progress=6


[t30] BATCH 6 SELESAI | agregasi diperbarui: 41
   2026-11-19 18:36:00 wikidatawiki edit= 361 bot%= 0.0
   2026-11-19 18:36:00 commonswiki edit= 217 bot%= 81.11
   2026-11-19 18:36:00 elwiktionary edit= 71 bot%= 98.59
   2026-11-19 18:36:00 enwiki edit= 58 bot%= 8.62
   2026-11-19 18:37:00 wikidatawiki edit= 32 bot%= 0.0


detik 60: aktif=True | progress=7
LAST PROGRESS: {
  "batchId": 6,
  "numInputRows": 398,
  "inputRowsPerSecond": 13.3,
  "processedRowsPerSecond": 187.4,
  "durationMs": {
    "addBatch": 1516,
    "commitOffsets": 318,
    "getBatch": 0,
    "latestOffset": 7,
    "queryPlanning": 30,
    "triggerExecution": 2124,
    "walCommit": 251
  },
  "eventTime": {
    "avg": "2026-11-19T11:36:55.558Z",
    "max": "2026-11-19T11:37:12.000Z",
    "min": "2026-11-19T11:36:43.000Z",
    "watermark": "2026-11-19T11:34:59.000Z"
  },
  "sources": [
    {
      "description": "KafkaV2[Subscribe[wiki.recentchange.raw]]",
      "startOffset": "{'wiki.recentchange.raw': {'0': 636, '1': 570, '2': 594, '3': 588}}",
      "endOffset": "{'wiki.recentchange.raw': {'0': 742, '1': 665, '2': 693, '3': 686}}",
      "latestOffset": "{'wiki.recentchange.raw': {'0': 127833, '1': 114825, '2': 119959, '3': 118004}}",
      "numInputRows": 398,
      "inputRowsPerSecond": 13.3,
      "processedRowsPerSecond": 187.4,

## SEL 13 — STOP trigger 30 detik

In [14]:
# SEL 13 — STOP 30s
last30 = stop_if_progress(q30, "t30")

STOP: t30 | aktif? False


## SEL 14 — 4B: Perbandingan terukur 10 vs 30 detik, ekspor CSV

In [15]:
# SEL 14 — RANGKUMAN REAL, BUKAN ANGKA CONTOH

def load_progress(tag):
    fn = BASE / f"progress_{tag}.jsonl"
    if not fn.exists():
        return []
    results = []
    for line in fn.read_text(encoding="utf-8").splitlines():
        try:
            results.append(json.loads(line))
        except ValueError:
            pass
    return results


def summarize(tag, trigger):
    all_p = load_progress(tag)
    useful = [p for p in all_p if p.get("numInputRows", 0) > 0]
    if not useful:
        print(f"WARNING: {tag} BELUM punya batch berisi data; tidak bisa dibandingkan secara valid.")
        return {"trigger_detik": trigger, "batch_berisi": 0,
                "inputRowsPerSecond": None, "processedRowsPerSecond": None,
                "batchDurationMs": None, "total_input_rows": 0}
    def mean(values):
        return round(statistics.mean(values), 2)
    return {
        "trigger_detik": trigger,
        "batch_berisi": len(useful),
        "inputRowsPerSecond": mean([float(p.get("inputRowsPerSecond", 0)) for p in useful]),
        "processedRowsPerSecond": mean([float(p.get("processedRowsPerSecond", 0)) for p in useful]),
        "batchDurationMs": mean([float(p.get("durationMs", {}).get("triggerExecution", 0))
                                 for p in useful]),
        "total_input_rows": sum(int(p.get("numInputRows", 0)) for p in useful),
    }

results_b = [summarize("t10", 10), summarize("t30", 30)]
for r in results_b:
    print(json.dumps(r, indent=2))

csv_file = BASE / "benchmark_10_vs_30.csv"
with csv_file.open("w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=list(results_b[0].keys()))
    writer.writeheader()
    writer.writerows(results_b)
print("HASIL CSV:", csv_file)
print("CATATAN: maxOffsetsPerTrigger=", MAX_OFFSETS_PER_TRIGGER,
      "membatasi laju pembacaan Kafka; bandingkan hanya batch valid.")

{
  "trigger_detik": 10,
  "batch_berisi": 16,
  "inputRowsPerSecond": 40.96,
  "processedRowsPerSecond": 170.55,
  "batchDurationMs": 2794.06,
  "total_input_rows": 6368
}
{
  "trigger_detik": 30,
  "batch_berisi": 7,
  "inputRowsPerSecond": 11.71,
  "processedRowsPerSecond": 170.03,
  "batchDurationMs": 2841.14,
  "total_input_rows": 2786
}
HASIL CSV: C:\Users\ASUS\UTS_Soal4_IP24_LokalTes\benchmark_10_vs_30.csv
CATATAN: maxOffsetsPerTrigger= 400 membatasi laju pembacaan Kafka; bandingkan hanya batch valid.


## SEL 15 — 4C: START ULANG 10 detik **dengan checkpoint yang sama**

**Jangan hapus** folder checkpoint `t10` dan DB `dashboard_t10.sqlite`. `q30` harus berhenti. Setelah restart, Kafka akan melanjutkan dari offset checkpoint, bukan otomatis membaca ulang dari `earliest`.

In [16]:
# SEL 15 — RESTART 4C
if q30.isActive:
    raise RuntimeError("q30 masih aktif. Jalankan Sel 13 dulu.")
if q10.isActive:
    raise RuntimeError("q10 lama masih aktif. Jalankan Sel 10 dulu.")
if before_restart is None:
    raise RuntimeError("Belum ada progress SEBELUM stop. Pengujian 4C belum valid.")
print("CHECKPOINT LAMA:", cp_path("t10"))
q10_restart = start_stream("t10", 10)  # CHECKPOINT PERSIS SAMA
print("RESTART aktif:", q10_restart.isActive)

CHECKPOINT LAMA: file:///C:/Users/ASUS/UTS_Soal4_IP24_LokalTes/checkpoints/t10
START: t10 trigger= 10 detik
CHECKPOINT: file:///C:/Users/ASUS/UTS_Soal4_IP24_LokalTes/checkpoints/t10 QUERY ID: 25790ec4-e16a-41e2-bfdb-bfb73c8a0edc
RESTART aktif: True


[t10] BATCH 29 SELESAI | agregasi diperbarui: 44
   2026-11-19 18:41:00 wikidatawiki edit= 274 bot%= 0.0
   2026-11-19 18:41:00 commonswiki edit= 179 bot%= 68.72
   2026-11-19 18:41:00 elwiktionary edit= 50 bot%= 100.0
   2026-11-19 18:42:00 wikidatawiki edit= 49 bot%= 2.04
   2026-11-19 18:41:00 enwiki edit= 39 bot%= 7.69


[t10] BATCH 30 SELESAI | agregasi diperbarui: 49
   2026-11-19 18:41:00 wikidatawiki edit= 303 bot%= 0.33
   2026-11-19 18:41:00 commonswiki edit= 205 bot%= 70.24
   2026-11-19 18:42:00 wikidatawiki edit= 76 bot%= 2.63
   2026-11-19 18:42:00 commonswiki edit= 62 bot%= 79.03
   2026-11-19 18:41:00 elwiktionary edit= 56 bot%= 100.0


[t10] BATCH 31 SELESAI | agregasi diperbarui: 44
   2026-11-19 18:41:00 wikidatawiki edit= 366 bot%= 0.27
   2026-11-19 18:41:00 commonswiki edit= 226 bot%= 69.03
   2026-11-19 18:42:00 wikidatawiki edit= 107 bot%= 1.87
   2026-11-19 18:42:00 commonswiki edit= 80 bot%= 77.5
   2026-11-19 18:41:00 elwiktionary edit= 64 bot%= 100.0


[t10] BATCH 32 SELESAI | agregasi diperbarui: 48
   2026-11-19 18:41:00 wikidatawiki edit= 386 bot%= 0.26
   2026-11-19 18:41:00 commonswiki edit= 243 bot%= 69.55
   2026-11-19 18:42:00 wikidatawiki edit= 170 bot%= 1.18
   2026-11-19 18:42:00 commonswiki edit= 105 bot%= 78.1
   2026-11-19 18:41:00 elwiktionary edit= 66 bot%= 100.0


[t10] BATCH 33 SELESAI | agregasi diperbarui: 37
   2026-11-19 18:42:00 wikidatawiki edit= 237 bot%= 0.84
   2026-11-19 18:42:00 commonswiki edit= 135 bot%= 77.78
   2026-11-19 18:42:00 elwiktionary edit= 53 bot%= 100.0
   2026-11-19 18:42:00 enwiki edit= 36 bot%= 0.0
   2026-11-19 18:43:00 wikidatawiki edit= 35 bot%= 0.0


[t10] BATCH 34 SELESAI | agregasi diperbarui: 49
   2026-11-19 18:42:00 wikidatawiki edit= 266 bot%= 0.75
   2026-11-19 18:42:00 commonswiki edit= 158 bot%= 79.11
   2026-11-19 18:43:00 wikidatawiki edit= 68 bot%= 0.0
   2026-11-19 18:42:00 elwiktionary edit= 56 bot%= 100.0
   2026-11-19 18:42:00 enwiki edit= 42 bot%= 0.0


[t10] BATCH 35 SELESAI | agregasi diperbarui: 46
   2026-11-19 18:42:00 wikidatawiki edit= 297 bot%= 0.67
   2026-11-19 18:42:00 commonswiki edit= 188 bot%= 77.66
   2026-11-19 18:43:00 wikidatawiki edit= 98 bot%= 0.0
   2026-11-19 18:43:00 commonswiki edit= 71 bot%= 73.24
   2026-11-19 18:42:00 elwiktionary edit= 67 bot%= 100.0


## SEL 16 — 4C: Pantau batch setelah restart (boleh diulang)

In [17]:
# SEL 16 — TUNGGU BEBERAPA BATCH RESTART
p_restart = show_metrics(q10_restart, "t10", watch_seconds=30)
# Query sengaja tetap aktif; bila belum ada progress, cek Spark UI lalu ulang sel ini.

[t10] BATCH 36 SELESAI | agregasi diperbarui: 54
   2026-11-19 18:42:00 wikidatawiki edit= 333 bot%= 1.2
   2026-11-19 18:42:00 commonswiki edit= 204 bot%= 77.45
   2026-11-19 18:43:00 wikidatawiki edit= 123 bot%= 0.0
   2026-11-19 18:43:00 commonswiki edit= 91 bot%= 74.73
   2026-11-19 18:42:00 elwiktionary edit= 71 bot%= 100.0


detik 5: aktif=True | progress=8


detik 10: aktif=True | progress=8


[t10] BATCH 37 SELESAI | agregasi diperbarui: 61
   2026-11-19 18:42:00 wikidatawiki edit= 350 bot%= 1.14
   2026-11-19 18:42:00 commonswiki edit= 209 bot%= 77.51
   2026-11-19 18:43:00 wikidatawiki edit= 152 bot%= 0.0
   2026-11-19 18:43:00 commonswiki edit= 116 bot%= 76.72
   2026-11-19 18:42:00 elwiktionary edit= 72 bot%= 100.0


detik 15: aktif=True | progress=9


detik 20: aktif=True | progress=9


[t10] BATCH 38 SELESAI | agregasi diperbarui: 51
   2026-11-19 18:43:00 wikidatawiki edit= 214 bot%= 0.0
   2026-11-19 18:42:00 commonswiki edit= 211 bot%= 77.73
   2026-11-19 18:43:00 commonswiki edit= 142 bot%= 78.17
   2026-11-19 18:44:00 wikidatawiki edit= 68 bot%= 0.0
   2026-11-19 18:42:00 enwiki edit= 54 bot%= 0.0


detik 25: aktif=True | progress=10


detik 30: aktif=True | progress=10
LAST PROGRESS: {
  "batchId": 38,
  "numInputRows": 398,
  "inputRowsPerSecond": 39.8,
  "processedRowsPerSecond": 222.0,
  "durationMs": {
    "addBatch": 1279,
    "commitOffsets": 257,
    "getBatch": 0,
    "latestOffset": 0,
    "queryPlanning": 17,
    "triggerExecution": 1793,
    "walCommit": 240
  },
  "eventTime": {
    "avg": "2026-11-19T11:43:41.668Z",
    "max": "2026-11-19T11:44:29.000Z",
    "min": "2026-11-19T11:42:57.000Z",
    "watermark": "2026-11-19T11:42:16.000Z"
  },
  "sources": [
    {
      "description": "KafkaV2[Subscribe[wiki.recentchange.raw]]",
      "startOffset": "{'wiki.recentchange.raw': {'0': 4028, '1': 3610, '2': 3762, '3': 3724}}",
      "endOffset": "{'wiki.recentchange.raw': {'0': 4134, '1': 3705, '2': 3861, '3': 3822}}",
      "latestOffset": "{'wiki.recentchange.raw': {'0': 127833, '1': 114825, '2': 119959, '3': 118004}}",
      "numInputRows": 398,
      "inputRowsPerSecond": 39.8,
      "processedRowsPerSecon

## SEL 17 — 4C: STOP dan audit offset + duplikat dashboard

In [18]:
# SEL 17 — AUDIT 4C
if not q10_restart.lastProgress:
    print("BELUM ADA progress setelah restart; tunggu dan ulang Sel 16 dulu.")
else:
    after_restart = stop_if_progress(q10_restart, "t10")
    db = db_path("t10")
    with sqlite3.connect(str(db)) as conn:
        unique_windows = conn.execute("SELECT COUNT(*) FROM dashboard").fetchone()[0]
        unique_batches = conn.execute("SELECT COUNT(*) FROM processed_batches").fetchone()[0]
        duplicates = conn.execute("""SELECT COUNT(*) FROM (
            SELECT wiki, window_start, window_end, COUNT(*) AS jumlah
            FROM dashboard GROUP BY wiki,window_start,window_end HAVING jumlah>1
        )""").fetchone()[0]
        dup_batches = conn.execute("""SELECT COUNT(*) FROM (
            SELECT batch_id, COUNT(*) AS jumlah FROM processed_batches
            GROUP BY batch_id HAVING jumlah>1
        )""").fetchone()[0]
    print("=== BUKTI OUTPUT DASHBOARD ===")
    print("Baris unik window/wiki:", unique_windows)
    print("Batch tersimpan:", unique_batches)
    print("Duplikat kunci window/wiki:", duplicates)
    print("Duplikat batch_id:", dup_batches)

    def parse_offset(obj):
        if not obj:
            return {}
        if isinstance(obj, str):
            try:
                obj = json.loads(obj)
            except ValueError:
                return {}
        return obj if isinstance(obj, dict) else {}

    prev_src = (before_restart.get("sources") or [{}])[0]
    # Cari progress pertama setelah restart berdasarkan batchId yang lebih tinggi.
    prev_bid = int(before_restart.get("batchId", -1))
    later = [p for p in load_progress("t10")
             if int(p.get("batchId", -1)) > prev_bid]
    later.sort(key=lambda x: int(x.get("batchId", -1)))
    if later:
        new_src = (later[0].get("sources") or [{}])[0]
        last_end = parse_offset(prev_src.get("endOffset")).get(KAFKA_TOPIC, {})
        next_start = parse_offset(new_src.get("startOffset")).get(KAFKA_TOPIC, {})
        print("=== OFFSET CHECKPOINT SEBELUM vs SETELAH ===")
        print("endOffset sebelum stop:", last_end)
        print("startOffset batch pertama sesudah restart:", next_start)
        if last_end and next_start:
            keys = sorted(set(last_end) | set(next_start))
            for k in keys:
                a, b = last_end.get(k), next_start.get(k)
                print(f"Partisi {k}: akhir={a} | awal_restart={b} | cocok={a == b}")
        else:
            print("Offset tidak lengkap pada progress; jangan klaim zero-skip tanpa bukti tambahan.")
    else:
        print("Belum ada batch baru yang TERCATAT setelah restart;")
        print("kesinambungan offset belum dapat dibuktikan dari recentProgress.")

    print("CATATAN: Duplikat 0 membuktikan keunikan output sink,")
    print("bukan otomatis membuktikan tak ada event terlambat yang dibuang watermark.")

STOP: t10 | aktif? False
=== BUKTI OUTPUT DASHBOARD ===
Baris unik window/wiki: 595
Batch tersimpan: 39
Duplikat kunci window/wiki: 0
Duplikat batch_id: 0
=== OFFSET CHECKPOINT SEBELUM vs SETELAH ===
endOffset sebelum stop: {}
startOffset batch pertama sesudah restart: {}
Offset tidak lengkap pada progress; jangan klaim zero-skip tanpa bukti tambahan.
CATATAN: Duplikat 0 membuktikan keunikan output sink,
bukan otomatis membuktikan tak ada event terlambat yang dibuang watermark.


In [19]:
# CEK OFFSET CHECKPOINT SOAL 4C
from pathlib import Path
import json

cp = BASE / "checkpoints" / "t10"
batch_sebelum = int(before_restart["batchId"])

print("=== AUDIT CHECKPOINT KAFKA ===")
print("Batch terakhir sebelum restart:", batch_sebelum)
print("Lokasi checkpoint:", cp)

for n in [batch_sebelum, batch_sebelum + 1]:
    f = cp / "offsets" / str(n)
    commit = cp / "commits" / str(n)

    print(f"\n--- BATCH {n} ---")
    print("File offset tersedia:", f.exists())
    print("Batch committed:", commit.exists())

    if f.exists():
        for line in f.read_text(encoding="utf-8").splitlines():
            try:
                data = json.loads(line)
                if isinstance(data, dict) and KAFKA_TOPIC in data:
                    print("Offset Kafka:", data[KAFKA_TOPIC])
            except json.JSONDecodeError:
                pass

print("\nSELESAI CEK CHECKPOINT")

=== AUDIT CHECKPOINT KAFKA ===
Batch terakhir sebelum restart: 28
Lokasi checkpoint: C:\Users\ASUS\UTS_Soal4_IP24_LokalTes\checkpoints\t10

--- BATCH 28 ---
File offset tersedia: True
Batch committed: True
Offset Kafka: {'0': 3074, '1': 2755, '2': 2871, '3': 2842}

--- BATCH 29 ---
File offset tersedia: True
Batch committed: True
Offset Kafka: {'0': 3180, '1': 2850, '2': 2970, '3': 2940}

SELESAI CEK CHECKPOINT


## SEL 18 — Bukti watermark, log, dan file output untuk screenshot

In [20]:
# SEL 18 — KUMPULKAN BUKTI
print("=== WATERMARK DAN ROW DROP (jika metrik tersedia) ===")
for tag in ("t10", "t30"):
    ps = load_progress(tag)
    if ps:
        p = ps[-1]
        print(tag, "eventTime:", p.get("eventTime"))
        for s in p.get("stateOperators") or []:
            print(tag, "numRowsDroppedByWatermark:", s.get("numRowsDroppedByWatermark"),
                  "numRowsTotal:", s.get("numRowsTotal"))
    else:
        print(tag, "belum ada progress")

print("\n=== OUTPUT DI PC MASTER ===")
for f in sorted(BASE.iterdir()):
    if f.is_file():
        print(f.name, "|", f.stat().st_size, "bytes")
print("\nScreenshot UTS:")
print("4A: Sel 4–7 dan output console batch Sel 8–9 + isi database dashboard.")
print("4B: Sel 9, 12, 14 (lastProgress dan benchmark_10_vs_30.csv).")
print("4C: Sel 10, 15–17 (checkpoint sama, offset, duplikat=0).")
if not SHARED_CP_URI:
    print("PERINGATAN FINAL: checkpoint yang digunakan lokal; belum bisa diklaim tahan pindah Worker.")

=== WATERMARK DAN ROW DROP (jika metrik tersedia) ===
t10 eventTime: {'avg': '2026-11-19T11:43:41.668Z', 'max': '2026-11-19T11:44:29.000Z', 'min': '2026-11-19T11:42:57.000Z', 'watermark': '2026-11-19T11:42:16.000Z'}
t10 numRowsDroppedByWatermark: 0 numRowsTotal: 157
t30 eventTime: {'avg': '2026-11-19T11:36:55.558Z', 'max': '2026-11-19T11:37:12.000Z', 'min': '2026-11-19T11:36:43.000Z', 'watermark': '2026-11-19T11:34:59.000Z'}
t30 numRowsDroppedByWatermark: 0 numRowsTotal: 126

=== OUTPUT DI PC MASTER ===
benchmark_10_vs_30.csv | 168 bytes
dashboard_t10.sqlite | 98304 bytes
dashboard_t30.sqlite | 36864 bytes
progress_t10.jsonl | 48689 bytes
progress_t30.jsonl | 12964 bytes

Screenshot UTS:
4A: Sel 4–7 dan output console batch Sel 8–9 + isi database dashboard.
4B: Sel 9, 12, 14 (lastProgress dan benchmark_10_vs_30.csv).
4C: Sel 10, 15–17 (checkpoint sama, offset, duplikat=0).
PERINGATAN FINAL: checkpoint yang digunakan lokal; belum bisa diklaim tahan pindah Worker.
